# 02. Cảm xúc & khía cạnh
Kết quả B5 (VADER, từ điển khía cạnh, TF-IDF), B4 (giờ chơi) và B8 (K-Means).

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Kết quả của profile local (python pipeline.py batch) nằm ở data/lake; đổi bằng biến LAKE_DIR
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LAKE = Path(os.getenv("LAKE_DIR", ROOT / "data" / "lake"))
CLEAN, CURATED = LAKE / "clean", LAKE / "curated"
read = pd.read_parquet

In [ ]:
aspects = read(CURATED / "aspect_stats")
agg = aspects.groupby("aspect")[["mention_count", "negative_count"]].sum()
agg["negative_ratio"] = agg.negative_count / agg.mention_count
agg.sort_values("negative_count").plot.barh(y="negative_count", figsize=(8, 4), title="Số lần nhắc trong review tiêu cực")
agg.sort_values("negative_ratio", ascending=False)

In [ ]:
joined = read(CURATED / "review_sentiment", columns=["recommendationid", "sentiment_score"]).merge(
    read(CLEAN / "reviews", columns=["recommendationid", "voted_up"]), on="recommendationid")
fig, ax = plt.subplots(figsize=(8, 4))
for label, d in joined.groupby("voted_up"):
    d.sentiment_score.plot.hist(bins=40, alpha=.5, ax=ax, label=f"voted_up={label}")
ax.legend(); ax.set_title("Điểm VADER theo nhãn Recommended")
print("Độ khớp dấu VADER với voted_up:", round(((joined.sentiment_score > 0) == joined.voted_up).mean(), 3))

In [ ]:
order = ["<2h", "2-10h", "10-50h", "50-200h", ">200h"]
b = read(CURATED / "playtime_buckets").groupby("playtime_bucket")[["positive_reviews", "total_reviews"]].sum().reindex(order)
(b.positive_reviews / b.total_reviews * 100).plot.bar(figsize=(8, 4), title="% tích cực theo giờ chơi khi review")

In [ ]:
retention = read(CURATED / "game_retention").query("total_reviews >= 50")
print("Game bị bỏ sớm nhiều nhất (early_negative_ratio cao):")
print(retention.nlargest(10, "early_negative_ratio")[["appid", "name", "early_negative_ratio", "positive_ratio_percent"]])
print("Game 'đáng tiền' nhất (giờ chơi trung vị / giá):")
retention.dropna(subset=["value_score"]).nlargest(10, "value_score")[["appid", "name", "price", "median_playtime_forever_hours", "value_score"]]

In [ ]:
kw = read(CURATED / "keywords")
top_game = kw.appid.value_counts().index[0]
kw[kw.appid == top_game].head(20)

In [ ]:
# Mô hình nhẹ: cụm hồ sơ trải nghiệm (B8 K-Means)
p = CURATED / "cluster_profiles"
read(p).sort_values("cluster") if p.exists() else "Chưa có cluster_profiles (B8 cần >= k game có >= 20 review)"